# How to Run Experiments with the Experiment Framework

This notebook demonstrates how to use the `Experiment` class to benchmark
shape matching methods on datasets.

We will:
1. Load the FAUST test set using `MeshDataset`
2. Create shape pairs using `PairsDataset`
3. Run a single experiment with `Experiment`
4. Compare multiple methods with `ExperimentSuite`
5. Analyze and save results

**Note:** For advanced features like grid search and configuration presets, see:
- [22_configuration_presets.ipynb](./22_configuration_presets.ipynb)
- [23_systematic_experiments.ipynb](./23_systematic_experiments.ipynb)

## Setup

In [ ]:
from geomfum.dataset.torch import MeshDataset, PairsDataset
from geomfum.experiment import Experiment, ExperimentConfig, ExperimentSuite
from geomfum.matcher import FeatureMatcher, FunctionalMapMatcher, MatcherConfig

## Load the FAUST Test Set

The FAUST dataset contains registered human body meshes. We load the test set
with spectral decomposition pre-computed (needed for functional maps).

In [2]:
# Path to the FAUST test set
dataset_dir = "../../../datasets/faust/test_set"

# Load meshes with spectral features
# spectral=True computes Laplacian eigenfunctions
# distances=True loads/computes geodesic distance matrices (for evaluation)
mesh_dataset = MeshDataset(
    dataset_dir=dataset_dir,
    spectral=True,
    distances=True,
    correspondences=True,  # FAUST has identity correspondences (same topology)
    k=30,  # Number of eigenfunctions
)

print(f"Loaded {len(mesh_dataset)} meshes")

Loaded 20 meshes


## Create Shape Pairs

We create pairs of shapes to match. For a quick demo, we use a small subset.

In [3]:
# Create pairs dataset
# pair_mode="all" creates all n*(n-1) pairs
# Use pairs_ratio to sample a subset for faster experimentation
pairs_dataset = PairsDataset(
    dataset=mesh_dataset,
    pairs_ratio=0.05,  # Use 5% of possible pairs for demo
)

print(f"Created {len(pairs_dataset)} pairs")

Created 380 pairs


## Single Experiment with Experiment

The `Experiment` class runs a single method on a dataset and evaluates it.

In [4]:
# Create matcher with custom config
config = MatcherConfig(
    spectrum_size=100,
    fmap_size=30,
    sdp_weight=1.0,
    lb_weight=1e-2,
)
matcher = FunctionalMapMatcher(config=config)

### Run the Experiment

The `Experiment` class handles:
- Iterating over all pairs
- Computing correspondences with the matcher
- Evaluating metrics (geodesic error, coverage, etc.)
- Aggregating results

In [5]:
# Configure the experiment
exp_config = ExperimentConfig(
    name="FAUST_FunctionalMap",
    bidirectional=False,  # Only compute A->B direction
    progress_bar=True,
)

# Create and run the experiment
experiment = Experiment(
    method=matcher,
    dataset=pairs_dataset,
    config=exp_config,
)

result = experiment.run()

Running FAUST_FunctionalMap:   1%|          | 4/380 [02:12<3:28:18, 33.24s/pair, geo_err=0.0094]


KeyboardInterrupt: 

### Analyze Results

In [ ]:
# View aggregated metrics
print("=" * 50)
print("Aggregated Metrics:")
print("=" * 50)
for metric, value in result.metrics.items():
    if not metric.endswith("_std"):
        std = result.metrics.get(f"{metric}_std", 0)
        print(f"{metric}: {value:.4f} ± {std:.4f}")

In [ ]:
# View per-pair metrics
import numpy as np

geodesic_errors = [
    m["geodesic_error"] for m in result.per_pair_metrics if "geodesic_error" in m
]

print("\nGeodesic Error Statistics:")
print(f"  Min:    {np.min(geodesic_errors):.4f}")
print(f"  Max:    {np.max(geodesic_errors):.4f}")
print(f"  Median: {np.median(geodesic_errors):.4f}")

In [ ]:
# Plot error distribution
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(geodesic_errors, bins=20, edgecolor="black", alpha=0.7)
plt.xlabel("Normalized Geodesic Error")
plt.ylabel("Count")
plt.title("Error Distribution")

plt.subplot(1, 2, 2)
plt.boxplot(geodesic_errors)
plt.ylabel("Normalized Geodesic Error")
plt.title("Error Boxplot")

plt.tight_layout()
plt.show()

### Save Results

You can save the results to a JSON file for later analysis.

In [ ]:
# Save results
# result.save("faust_fmatcher_results.json")

# Load results later
# from geomfum.experiment import ExperimentResult
# loaded_result = ExperimentResult.load("faust_fmatcher_results.json")

## Compare Multiple Methods with ExperimentSuite

The `ExperimentSuite` class makes it easy to compare multiple methods side-by-side.

In [ ]:
# Define methods to compare
methods = {
    "FunctionalMap": FunctionalMapMatcher(),
    "Feature": FeatureMatcher(),
}

# Create and run suite
suite = ExperimentSuite(methods, pairs_dataset)
all_results = suite.run()

In [ ]:
# Print comparison table
suite.print_comparison(metrics=["geodesic_error", "coverage"])

In [ ]:
# Save all results
# suite.save_all("results/")

## Next Steps

For more advanced experimentation patterns:

- **[22_configuration_presets.ipynb](./22_configuration_presets.ipynb)** - Using preset configurations
- **[23_systematic_experiments.ipynb](./23_systematic_experiments.ipynb)** - Grid search and parameter exploration
- **[19_matcher.ipynb](./19_matcher.ipynb)** - Customizing matcher configurations